# OpenCode Go

> Route OpenCode Go models to FastLLM's existing protocol adapters.

OpenCode Go serves models through more than one API surface. This notebook keeps the routing decision in one explicit table, allowing generic completion code to select an existing adapter without provider-specific branches.

In [ ]:
#| default_exp opencode

In [ ]:
#| hide
from fastcore.test import *

## Routes

A route records the protocol adapter, API root, and credential environment variable for one model, as a tuple that `mk_client` unpacks directly. The initial table contains only the Chat Completions route identified in the proposal. Add models only after their served API surface has been verified.

In [ ]:
opencode_go_base = 'https://opencode.ai/zen/go/v1'
opencode_go_anthropic_base = 'https://opencode.ai/zen/go'
opencode_api_key_env = 'OPENCODE_API_KEY'
opencode_user_agent = 'fastllm'

_opencode_routes = {
    **dict.fromkeys(['grok-4.6', 'gpt-5.6-luna', 'muse-spark-1.3-contributor', 'muse-spark-1.2-contributor'],
                     ('openai', opencode_go_base, opencode_api_key_env)),
    **dict.fromkeys(['glm-5.3-flash', 'glm-5.3', 'glm-5.2', 'glm-5.1',
                     'kimi-k3', 'kimi-k2.7-code', 'kimi-k2.6', 'longcat-2.0',
                     'deepseek-v4.1-flash', 'deepseek-v4-pro', 'deepseek-v4-flash', 'deepseek-v4-flash-vision-exp',
                     'mimo-v2.5', 'mimo-v2.5-pro', 'hy4-preview', 'hy3'],
                     ('openai_chat', opencode_go_base, opencode_api_key_env)),
    **dict.fromkeys(['minimax-m3', 'minimax-m2.7', 'qwen3.8-max', 'qwen3.8-flash',
                     'qwen3.7-max', 'qwen3.7-plus', 'qwen3.6-plus'],
                     ('anthropic', opencode_go_anthropic_base, opencode_api_key_env)),
}

def resolve_opencode_go(model):
    "Return the verified OpenCode Go '(api_name, base_url, api_key_env)' route for `model`."
    try: return _opencode_routes[model]
    except KeyError: raise ValueError(f'Unknown OpenCode Go model: {model}') from None

The resolver returns the route tuple itself; tuples are immutable, so a caller cannot mutate the authoritative route table through it.

In [ ]:
route = resolve_opencode_go('kimi-k3')
test_eq(route, ('openai_chat', "https://opencode.ai/zen/go/v1", "OPENCODE_API_KEY"))
route

('openai_chat', 'https://opencode.ai/zen/go/v1', 'OPENCODE_API_KEY')

The route table mirrors OpenCode Go's current model list. Responses and Chat routes use the shared `/v1` base because their adapters append `/responses` or `/chat/completions`. Anthropic routes deliberately use the base without `/v1`, because the Anthropic adapter already appends `/v1/messages`. `minimax-m2.5` appears in OpenCode's endpoint table but not its current-model list, so it is intentionally excluded.

In [ ]:
test_eq(len(_opencode_routes), 27)
test_eq({route[0] for route in _opencode_routes.values()}, {'openai', 'openai_chat', 'anthropic'})
test_eq({model for model, route in _opencode_routes.items() if route[0] == 'openai'},
        {'grok-4.6', 'gpt-5.6-luna', 'muse-spark-1.3-contributor', 'muse-spark-1.2-contributor'})
test_eq({model for model, route in _opencode_routes.items() if route[0] == 'anthropic'},
        {'minimax-m3', 'minimax-m2.7', 'qwen3.8-max', 'qwen3.8-flash', 'qwen3.7-max', 'qwen3.7-plus', 'qwen3.6-plus'})
test_eq({route[1] for route in _opencode_routes.values() if route[0] == 'anthropic'}, {opencode_go_anthropic_base})
test_eq({route[2] for route in _opencode_routes.values()}, {opencode_api_key_env})
_opencode_routes

{'grok-4.6': ('openai', 'https://opencode.ai/zen/go/v1', 'OPENCODE_API_KEY'),
 'gpt-5.6-luna': ('openai',
  'https://opencode.ai/zen/go/v1',
  'OPENCODE_API_KEY'),
 'muse-spark-1.3-contributor': ('openai',
  'https://opencode.ai/zen/go/v1',
  'OPENCODE_API_KEY'),
 'muse-spark-1.2-contributor': ('openai',
  'https://opencode.ai/zen/go/v1',
  'OPENCODE_API_KEY'),
 'glm-5.3-flash': ('openai_chat',
  'https://opencode.ai/zen/go/v1',
  'OPENCODE_API_KEY'),
 'glm-5.3': ('openai_chat',
  'https://opencode.ai/zen/go/v1',
  'OPENCODE_API_KEY'),
 'glm-5.2': ('openai_chat',
  'https://opencode.ai/zen/go/v1',
  'OPENCODE_API_KEY'),
 'glm-5.1': ('openai_chat',
  'https://opencode.ai/zen/go/v1',
  'OPENCODE_API_KEY'),
 'kimi-k3': ('openai_chat',
  'https://opencode.ai/zen/go/v1',
  'OPENCODE_API_KEY'),
 'kimi-k2.7-code': ('openai_chat',
  'https://opencode.ai/zen/go/v1',
  'OPENCODE_API_KEY'),
 'kimi-k2.6': ('openai_chat',
  'https://opencode.ai/zen/go/v1',
  'OPENCODE_API_KEY'),
 'longcat-2.0': ('o

In [ ]:
with expect_fail(ValueError, 'Unknown OpenCode Go model'):
    resolve_opencode_go('unverified-model')

## Session headers

OpenCode requires each client to identify itself with its own `User-Agent` and to send a stable `x-opencode-session` header within a conversation. The helper supplies the FastLLM user agent by default, allows a caller to replace it, and applies the session identity last so a stale or conflicting session value cannot reach the request.

In [ ]:
def opencode_headers(session_id, headers=None):
    "Return headers with a validated OpenCode session identity."
    if not isinstance(session_id, str) or not session_id.strip():
        raise ValueError('OpenCode session_id must be a non-empty string')
    return {'User-Agent': opencode_user_agent, **(headers or {}), 'x-opencode-session': session_id}

In [ ]:
test_eq(opencode_headers('session-1'),
        {'User-Agent': opencode_user_agent, 'x-opencode-session': 'session-1'})
headers = opencode_headers('session-1',
        {'User-Agent': 'my-coding-agent/1.0', 'x-client': 'fastllm', 'x-opencode-session': 'stale'})
test_eq(headers, {'User-Agent': 'my-coding-agent/1.0', 'x-client': 'fastllm', 'x-opencode-session': 'session-1'})
headers

{'User-Agent': 'my-coding-agent/1.0',
 'x-client': 'fastllm',
 'x-opencode-session': 'session-1'}

In [ ]:
with expect_fail(ValueError, 'non-empty'):
    opencode_headers('')

The next change is to connect `resolve_opencode_go` to vendor resolution in `06_acomplete.ipynb`. Model registrations, limits, and pricing follow once the OpenCode catalog and rates are confirmed.

In [ ]:
#|hide
#|eval: false
import nbdev
nbdev.nbdev_export()